# Benchmarks: polars-uuid plugin vs. naive Python UUID generation

Compares three ways of adding a UUID column to a dataframe of `n` rows:

1. **polars + polars_uuid** — the Rust plugin's native expressions
2. **polars + naive** — `pl.Expr.map_elements` calling the stdlib `uuid` module once per row
3. **pandas + naive** — a list comprehension over the stdlib `uuid` module (the fastest common
   pandas idiom for this — faster than `Series.apply`, which pays extra per-element overhead)

Two functions are benchmarked: `uuid4` (pure random generation, no input data needed) and
`uuid5` (deterministic, derived from a string column — more representative of turning an
existing natural key into a stable UUID).

In [1]:
import timeit
import uuid

import pandas as pd
import polars as pl

import polars_uuid

SIZES = [1_000, 10_000, 100_000, 1_000_000]
REPEATS = 3  # take the minimum of this many runs, to reduce noise from other work on the machine


def timed(fn) -> float:
    """Best-of-`REPEATS` wall-clock seconds for a single call to `fn()`."""
    return min(timeit.repeat(fn, repeat=REPEATS, number=1))

## `uuid4`: random generation

In [2]:
def bench_uuid4(n: int) -> dict[str, float]:
    pldf = pl.DataFrame({"row": range(n)})
    pddf = pd.DataFrame({"row": range(n)})

    return {
        "n": n,
        "polars + polars_uuid": timed(
            lambda: pldf.with_columns(polars_uuid.uuid4("row").alias("id"))
        ),
        "polars + naive": timed(
            lambda: pldf.with_columns(
                pl.col("row")
                .map_elements(lambda _: str(uuid.uuid4()), return_dtype=pl.String)
                .alias("id")
            )
        ),
        "pandas + naive": timed(
            lambda: pddf.assign(
                id=[str(uuid.uuid4()) for _ in range(len(pddf))]
            )
        ),
    }


uuid4_results = pl.DataFrame([bench_uuid4(n) for n in SIZES])
uuid4_results

n,polars + polars_uuid,polars + naive,pandas + naive
i64,f64,f64,f64
1000,0.00055,0.004602,0.003977
10000,0.004436,0.040974,0.038975
100000,0.043546,0.409312,0.388192
1000000,0.437204,4.085945,3.979951


## `uuid5`: deterministic, derived from a string column

In [3]:
def bench_uuid5(n: int) -> dict[str, float]:
    names = [f"user-{i}@example.com" for i in range(n)]
    pldf = pl.DataFrame({"name": names})
    pddf = pd.DataFrame({"name": names})

    return {
        "n": n,
        "polars + polars_uuid": timed(
            lambda: pldf.with_columns(
                polars_uuid.uuid5("name", namespace=uuid.NAMESPACE_DNS).alias("id")
            )
        ),
        "polars + naive": timed(
            lambda: pldf.with_columns(
                pl.col("name")
                .map_elements(
                    lambda v: str(uuid.uuid5(uuid.NAMESPACE_DNS, v)),
                    return_dtype=pl.String,
                )
                .alias("id")
            )
        ),
        "pandas + naive": timed(
            lambda: pddf.assign(
                id=[str(uuid.uuid5(uuid.NAMESPACE_DNS, v)) for v in pddf["name"]]
            )
        ),
    }


uuid5_results = pl.DataFrame([bench_uuid5(n) for n in SIZES])
uuid5_results

n,polars + polars_uuid,polars + naive,pandas + naive
i64,f64,f64,f64
1000,0.000363,0.005989,0.005724
10000,0.002392,0.044674,0.046925
100000,0.019977,0.440299,0.490298
1000000,0.199385,4.432231,4.926365


## Speedup vs. naive polars, at the largest size

In [4]:
def summarize(results: pl.DataFrame, label: str) -> None:
    row = results.filter(pl.col("n") == SIZES[-1]).row(0, named=True)
    plugin, naive_pl, naive_pd = (
        row["polars + polars_uuid"],
        row["polars + naive"],
        row["pandas + naive"],
    )
    print(f"{label} @ n={SIZES[-1]:,}")
    print(f"  polars_uuid is {naive_pl / plugin:6.1f}x faster than naive polars (map_elements)")
    print(f"  polars_uuid is {naive_pd / plugin:6.1f}x faster than naive pandas (list comprehension)")


summarize(uuid4_results, "uuid4")
print()
summarize(uuid5_results, "uuid5")

uuid4 @ n=1,000,000
  polars_uuid is    9.3x faster than naive polars (map_elements)
  polars_uuid is    9.1x faster than naive pandas (list comprehension)

uuid5 @ n=1,000,000
  polars_uuid is   22.2x faster than naive polars (map_elements)
  polars_uuid is   24.7x faster than naive pandas (list comprehension)
